In [1]:
import mne
import numpy as np
import os
from scipy.signal import stft
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

data_path = r"C:\Users\hiro2\Documents\EEG_project\data\MUSIN-G"
save_path = r"C:\Users\hiro2\Documents\EEG_project\data\spectrograms"
os.makedirs(save_path, exist_ok=True)

# Muse S Athena対応チャンネル
MUSE_CH = [50, 26, 88, 7]
MUSE_CH_NAMES = ['AF7', 'AF8', 'TP9', 'TP10']
SF = 250
NPERSEG = 128
FREQ_MAX = 40

for sub_id in range(1, 21):
    for ses_id in range(1, 13):
        set_path = os.path.join(
            data_path,
            f"sub-{sub_id:03d}",
            f"ses-{ses_id:02d}",
            "eeg",
            f"sub-{sub_id:03d}_ses-{ses_id:02d}_task-MusicListening_run-{ses_id}_eeg.set"
        )
        if not os.path.exists(set_path):
            continue

        try:
            raw = mne.io.read_raw_eeglab(set_path, preload=True, verbose=False)
            raw.filter(1., 40., fir_window='hamming', verbose=False)
            data = raw.get_data()[MUSE_CH]  # (4ch, timepoints)

            spec_list = []
            for ch_data in data:
                f, t, Zxx = stft(ch_data, fs=SF, nperseg=NPERSEG)
                freq_idx = f <= FREQ_MAX
                power = np.abs(Zxx[freq_idx])
                power = np.log1p(power)
                spec_list.append(power)

            # (4ch, freq_bins, time_bins)
            spec = np.stack(spec_list, axis=0)

            fname = f"sub{sub_id:03d}_ses{ses_id:02d}.npy"
            np.save(os.path.join(save_path, fname), spec)

            print(f"✓ sub-{sub_id:03d} ses-{ses_id:02d} shape={spec.shape}")

        except Exception as e:
            print(f"✗ sub-{sub_id:03d} ses-{ses_id:02d}: {e}")

print(f"\n完了！{save_path} に保存")

# サンプル可視化
sample = np.load(os.path.join(save_path, "sub001_ses01.npy"))
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, ax in enumerate(axes):
    ax.imshow(sample[i], aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(MUSE_CH_NAMES[i])
    ax.set_xlabel('Time')
    ax.set_ylabel('Freq bin')
plt.tight_layout()
plt.savefig(r"C:\Users\hiro2\Documents\EEG_project\data\sample_spectrogram.png", dpi=150)
print("サンプル画像保存完了")

✓ sub-001 ses-01 shape=(4, 21, 533)
✓ sub-001 ses-02 shape=(4, 21, 489)
✓ sub-001 ses-03 shape=(4, 21, 559)
✓ sub-001 ses-04 shape=(4, 21, 476)
✓ sub-001 ses-05 shape=(4, 21, 527)
✓ sub-001 ses-06 shape=(4, 21, 431)
✓ sub-001 ses-07 shape=(4, 21, 496)
✓ sub-001 ses-08 shape=(4, 21, 516)
✓ sub-001 ses-09 shape=(4, 21, 535)
✓ sub-001 ses-10 shape=(4, 21, 546)
✓ sub-001 ses-11 shape=(4, 21, 483)
✓ sub-001 ses-12 shape=(4, 21, 501)
✓ sub-002 ses-01 shape=(4, 21, 2127)
✓ sub-002 ses-02 shape=(4, 21, 1953)
✓ sub-002 ses-03 shape=(4, 21, 2232)
✓ sub-002 ses-04 shape=(4, 21, 1899)
✓ sub-002 ses-05 shape=(4, 21, 2104)
✓ sub-002 ses-06 shape=(4, 21, 1721)
✓ sub-002 ses-07 shape=(4, 21, 1978)
✓ sub-002 ses-08 shape=(4, 21, 2061)
✓ sub-002 ses-09 shape=(4, 21, 2135)
✓ sub-002 ses-10 shape=(4, 21, 2181)
✓ sub-002 ses-11 shape=(4, 21, 1926)
✓ sub-002 ses-12 shape=(4, 21, 1999)
✓ sub-003 ses-01 shape=(4, 21, 533)
✓ sub-003 ses-02 shape=(4, 21, 489)
✓ sub-003 ses-03 shape=(4, 21, 559)
✓ sub-003 ses-04